In [47]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    normalize_text,
    save_parquet,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
LIFECYCLE_EVENTS_PATH = SILVER_BOE_AI_DIR / "lifecycle_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
ASSET_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_mentions.parquet"
ASSET_TECHNOLOGIES_PATH = SILVER_BOE_AI_DIR / "asset_technologies.parquet"
ASSET_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "asset_participants.parquet"
ASSET_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "asset_locations.parquet"
ASSET_ALIASES_PATH = SILVER_BOE_AI_DIR / "asset_aliases.parquet"
ASSET_RELATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_relation_mentions.parquet"
GROUPING_CANDIDATES_PATH = SILVER_BOE_AI_DIR / "grouping_candidates.parquet"


# GOLD
PROJECT_GROUPS_PATH = GOLD_DIR / "project_groups.parquet"
PROJECT_ASSETS_PATH = GOLD_DIR / "project_assets.parquet"
PROJECT_TIMELINE_PATH = GOLD_DIR / "project_timeline.parquet"
PROJECT_STATUS_PATH = GOLD_DIR / "project_status.parquet"

## 1. Instrucciones por fases

In [48]:
BASE_EXTRACTION_RULES = """
Reglas generales:
- Extrae únicamente información explícitamente contenida en el título o en el texto del documento.
- No inventes, completes ni corrijas datos por conocimiento externo.
- Si un dato no aparece, usa null, lista vacía o no_consta, según corresponda al esquema.
- Toda información relevante debe estar respaldada por evidencia textual.
- No asignes identificadores definitivos de proyecto.
- Usa solo identificadores internos al documento cuando el esquema lo requiera.
"""

In [49]:
SPANISH_NUMBER_RULES = """
Interpreta los números con formato español:
- "31,172 MW" equivale a 31.172 MW.
- "28.000 kW" equivale a 28000 kW.
- Cuando el texto incluya potencia unitaria y número de equipos, calcula la potencia total normalizada si la equivalencia es explícita.
- Si existe discrepancia aparente entre el cálculo explícito y una cifra textual, no afirmes que el BOE contiene una errata.
- Conserva la discrepancia en power_normalization_note.
"""

In [50]:
CLASSIFICATION_INSTRUCTIONS = f"""
Eres un clasificador documental del BOE sobre proyectos energéticos.

Devuelve exclusivamente JSON válido conforme al esquema DocumentClassification.

Objetivo:
Determinar si el documento es relevante para el seguimiento de proyectos energéticos concretos.

{BASE_EXTRACTION_RULES}

Criterios:
- Marca es_relevante_para_proyecto = true si el documento trata sobre un proyecto energético identificable.
- Marca es_relevante_para_proyecto = false si trata sobre normas generales, convocatorias, ayudas, planificación abstracta o información no asociada a un proyecto concreto.
- Usa relevancia_energetica = relevante, no_relevante o dudoso.
- relevance_reason debe explicar brevemente la decisión.
- evidence debe contener una cita breve del título o texto.
"""

In [51]:
ADMIN_EVENT_INSTRUCTIONS = f"""
Eres un extractor de acciones administrativas del BOE sobre proyectos energéticos.

Devuelve exclusivamente JSON válido conforme al esquema AdministrativeActionExtraction.

Objetivo:
Extraer los actos administrativos publicados en el BOE.

{BASE_EXTRACTION_RULES}

Reglas:
- Extrae una o varias administrative_actions si el documento contiene varios actos administrativos.
- Cada administrative_action debe tener stage, decision y evidence.
- No extraigas activos.
- No extraigas promotores.
- No extraigas municipios.
- No incluyas antecedentes menores salvo que sean necesarios para interpretar el acto principal.

Ejemplos:
- Si el documento formula una DIA, usa stage = declaracion_impacto_ambiental y decision = formulado o favorable, según el texto.
- Si el documento otorga AAP y AAC, crea dos administrative_actions.
"""

In [52]:
LIFECYCLE_EVENT_INSTRUCTIONS = f"""
Eres un extractor de eventos de ciclo de vida de proyectos energéticos.

Devuelve exclusivamente JSON válido conforme al esquema LifecycleEventExtraction.

Objetivo:
Determinar qué tipo de evolución del proyecto representa el documento.

{BASE_EXTRACTION_RULES}

Reglas:
- event_type debe describir la evolución material o funcional del proyecto:
  new_project, hybridization, modification, expansion, repowering, storage_addition, evacuation_infrastructure, ownership_change, other o unknown.
- No uses event_type para representar el trámite jurídico. Los trámites van en administrative_actions.
- event_summary debe contener una frase breve y no nula.
- evidence debe contener una cita breve del título o texto.

Ejemplo:
Si el documento formula un informe ambiental para una planta fotovoltaica que hibrida un parque eólico existente:
- event_type = hybridization.
"""

In [53]:
ASSET_INSTRUCTIONS = f"""
Eres un extractor de activos energéticos del BOE.

Devuelve exclusivamente JSON válido conforme al esquema AssetExtraction.

Objetivo:
Extraer los activos energéticos principales afectados por el documento.

{BASE_EXTRACTION_RULES}

{SPANISH_NUMBER_RULES}

Activos:
- Extrae nuevas instalaciones de generación.
- Extrae instalaciones existentes afectadas por hibridación, modificación, ampliación o repotenciación.
- Extrae sistemas de almacenamiento.
- Extrae infraestructuras energéticas solo si son el objeto principal del documento.

Restricciones:
- No extraigas líneas, subestaciones, centros de seccionamiento ni puntos de conexión como assets si solo aparecen como infraestructura auxiliar.
- Usa local_asset_id internos al documento: asset_1, asset_2, asset_3.
- No extraigas participantes ni municipios en esta fase, salvo si el esquema AssetExtraction los incluye expresamente.

Hibridación:
- Crea un asset para la nueva instalación.
- Crea otro asset para el activo existente afectado si aparece en el texto.
- Marca la nueva instalación como new_asset.
- Marca el activo previo como existing_asset.
"""

In [54]:
ASSET_RELATION_INSTRUCTIONS = f"""
Eres un extractor de relaciones entre activos energéticos.

Devuelve exclusivamente JSON válido conforme al esquema AssetRelationExtraction.

Objetivo:
Extraer relaciones explícitas entre activos ya identificados en el documento.

{BASE_EXTRACTION_RULES}

Reglas:
- Usa asset_relations únicamente entre activos energéticos principales.
- No describas toda la red de evacuación, conexión o acceso.
- Usa los local_asset_id proporcionados en el contexto.
- No inventes relaciones si el texto no las expresa.

Tipos de relación:
- hybridizes_with
- adds_technology_to
- modifies
- expands
- repowers
- adds_storage_to
- same_project_group_as
- associated_with
"""

In [55]:
PARTICIPANT_INSTRUCTIONS = f"""
Eres un extractor de participantes empresariales o administrativos en documentos del BOE sobre proyectos energéticos.

Devuelve exclusivamente JSON válido conforme al esquema ParticipantExtraction.

Objetivo:
Extraer promotores, copromotores, titulares, operadores u otras entidades participantes explícitamente mencionadas.

{BASE_EXTRACTION_RULES}

Reglas:
- Extrae solo entidades jurídicas o administrativas explícitas.
- Puede haber varios participantes.
- No asumas que el promotor de un activo es también promotor de otro si el texto no lo dice.
- Conserva la denominación literal.
- No inventes CIF/NIF.
- No resuelvas entidades por conocimiento externo.
- Usa evidence textual breve para cada participante.
"""

In [56]:
LOCATION_INSTRUCTIONS = f"""
Eres un extractor de localizaciones administrativas de activos energéticos en documentos del BOE.

Devuelve exclusivamente JSON válido conforme al esquema LocationExtraction.

Objetivo:
Extraer municipios, provincias y comunidades autónomas asociados explícitamente a activos energéticos.

{BASE_EXTRACTION_RULES}

Municipios:
- Extrae el nombre del municipio mencionado en el documento.
- Extrae provincia y comunidad autónoma solo si aparecen en el documento.
- Llama siempre a la herramienta resolve_municipality.
- Utiliza exclusivamente el resultado devuelto por resolve_municipality para completar municipio oficial, provincia, comunidad autónoma y códigos INE.
- No inventes códigos INE ni divisiones administrativas.
- No incluyas municipios mencionados solo en direcciones postales, sedes sociales o antecedentes no relacionados con la ubicación del activo.
"""

In [57]:
GROUPING_INSTRUCTIONS = f"""
Eres un extractor de candidatos de agrupación para publicaciones BOE sobre proyectos energéticos.

Devuelve exclusivamente JSON válido conforme al esquema GroupingCandidateExtraction.

Objetivo:
Extraer cadenas útiles para agrupar publicaciones futuras sobre el mismo proyecto o complejo energético.

{BASE_EXTRACTION_RULES}

Reglas:
- Incluye nombres de activos principales.
- Incluye aliases relevantes.
- Incluye nombres base compartidos si aparecen claramente en el texto.
- Incluye municipios, provincias y promotores si constan.
- No incluyas subestaciones, líneas ni puntos de conexión auxiliares salvo que sean el objeto principal del documento.
- No inventes project_group_id definitivo.
"""

## 2. Contratos de salida faseados

Qué quiero saber de cada publicación?

In [58]:
# ============================================================
# 1. Clasificación general del documento
# ============================================================

class RelevanciaEnergetica(str, Enum):
    RELEVANTE = "relevante"
    NO_RELEVANTE = "no_relevante"
    DUDOSO = "dudoso"


class DocumentClassification(BaseModel):
    identificador_boe: str
    fecha_publicacion: date | None = None

    relevancia_energetica: RelevanciaEnergetica
    es_relevante_para_proyecto: bool
    relevance_reason: str | None = None
    evidence: str | None = None

In [59]:
# ============================================================
# 2. Tecnologías y almacenamiento
# ============================================================

class TechnologyType(str, Enum):
    FOTOVOLTAICA = "fotovoltaica"
    EOLICA = "eolica"
    TERMOSOLAR = "termosolar"
    HIDROELECTRICA = "hidroelectrica"
    GEOTERMICA = "geotermica"
    BIOMASA = "biomasa"
    BIOGAS = "biogas"
    HIDROGENO_VERDE = "hidrogeno_verde"
    ALMACENAMIENTO = "almacenamiento"
    OTRA = "otra"
    DESCONOCIDA = "desconocida"


class Technology(BaseModel):
    technology_type: TechnologyType
    installed_power_mw: float | None = None
    peak_power_mwp: float | None = None
    description: str | None = None
    power_normalization_note: str | None = None


class StorageSystem(BaseModel):
    exists: bool = False
    power_mw: float | None = None
    capacity_mwh: float | None = None
    description: str | None = None

In [60]:
# ============================================================
# 3. Localización normalizada mediante INE
# ============================================================

class MunicipalityResolutionStatus(str, Enum):
    RESOLVED = "resolved"
    AMBIGUOUS = "ambiguous"
    NOT_FOUND = "not_found"


class MunicipalityLocation(BaseModel):
    ine_municipality_code: str
    municipality: str
    ine_province_code: str
    province: str
    ine_autonomous_community_code: str | None = None
    autonomous_community: str | None = None


class MunicipalityLookupResult(BaseModel):
    query: str
    province_hint: str | None = None
    autonomous_community_hint: str | None = None

    resolution_status: MunicipalityResolutionStatus
    resolved: MunicipalityLocation | None = None
    candidates: list[MunicipalityLocation] = Field(default_factory=list)

    matched_by: str | None = None
    reason: str | None = None

In [61]:
# ============================================================
# 4. Procedimiento administrativo
# ============================================================

class ProcedureStage(str, Enum):
    SOLICITUD_TRAMITACION = "solicitud_tramitacion"
    SOLICITUD_TRAMITACION_AMBIENTAL = "solicitud_tramitacion_ambiental"
    SUBSANACION_DOCUMENTACION = "subsanacion_documentacion"
    VERIFICACION_REQUISITOS_TRAMITACION = "verificacion_requisitos_tramitacion"

    INFORMACION_PUBLICA = "informacion_publica"

    DECLARACION_IMPACTO_AMBIENTAL = "declaracion_impacto_ambiental"
    INFORME_DETERMINACION_AFECCION_AMBIENTAL = "informe_determinacion_afeccion_ambiental"

    AUTORIZACION_ADMINISTRATIVA_PREVIA = "autorizacion_administrativa_previa"
    AUTORIZACION_ADMINISTRATIVA_CONSTRUCCION = "autorizacion_administrativa_construccion"
    AUTORIZACION_EXPLOTACION = "autorizacion_explotacion"

    DECLARACION_UTILIDAD_PUBLICA = "declaracion_utilidad_publica"
    EXPROPIACION_FORZOSA = "expropiacion_forzosa"
    RELACION_BIENES_DERECHOS_AFECTADOS = "relacion_bienes_derechos_afectados"
    LEVANTAMIENTO_ACTAS_PREVIAS_OCUPACION = "levantamiento_actas_previas_ocupacion"
    ACTAS_OCUPACION = "actas_ocupacion"

    MODIFICACION = "modificacion"
    ARCHIVO_EXPEDIENTE = "archivo_expediente"
    DESISTIMIENTO = "desistimiento"
    INADMISION = "inadmision"

    OTRO = "otro"
    NO_CONSTA = "no_consta"


class ProcedureDecision(str, Enum):
    SOLICITADO = "solicitado"
    SUBSANADO = "subsanado"
    REQUISITOS_VERIFICADOS = "requisitos_verificados"

    FORMULADO = "formulado"
    FAVORABLE = "favorable"
    DESFAVORABLE = "desfavorable"
    SOMETIDO_EIA_ORDINARIA = "sometido_eia_ordinaria"
    NO_SOMETIDO_EIA_ORDINARIA = "no_sometido_eia_ordinaria"

    SOMETIDO_INFORMACION_PUBLICA = "sometido_informacion_publica"
    CONVOCADO = "convocado"

    AUTORIZADO = "autorizado"
    APROBADO = "aprobado"
    DENEGADO = "denegado"

    DECLARADO_UTILIDAD_PUBLICA = "declarado_utilidad_publica"

    MODIFICADO = "modificado"
    PRORROGADO = "prorrogado"

    ARCHIVADO = "archivado"
    DESISTIDO = "desistido"
    INADMITIDO = "inadmitido"

    NO_CONSTA = "no_consta"


class AdministrativeAction(BaseModel):
    stage: ProcedureStage = ProcedureStage.NO_CONSTA
    decision: ProcedureDecision = ProcedureDecision.NO_CONSTA
    evidence: str | None = None


class AdministrativeActionExtraction(BaseModel):
    administrative_actions: list[AdministrativeAction] = Field(default_factory=list)

In [62]:
# ============================================================
# 5. Evento de ciclo de vida
# ============================================================

class LifecycleEventType(str, Enum):
    NEW_PROJECT = "new_project"
    HYBRIDIZATION = "hybridization"
    MODIFICATION = "modification"
    EXPANSION = "expansion"
    REPOWERING = "repowering"
    STORAGE_ADDITION = "storage_addition"
    EVACUATION_INFRASTRUCTURE = "evacuation_infrastructure"
    OWNERSHIP_CHANGE = "ownership_change"
    OTHER = "other"
    UNKNOWN = "unknown"


class LifecycleEventExtraction(BaseModel):
    event_type: LifecycleEventType = LifecycleEventType.UNKNOWN
    event_summary: str | None = None
    evidence: str | None = None

In [63]:
# ============================================================
# 6. Participantes
# ============================================================

class ParticipantRole(str, Enum):
    PROMOTER = "promoter"
    CO_PROMOTER = "co_promoter"
    OWNER = "owner"
    OPERATOR = "operator"
    GRID_OWNER = "grid_owner"
    ADMINISTRATION = "administration"
    UNKNOWN = "unknown"


class ProjectParticipant(BaseModel):
    name: str
    role: ParticipantRole = ParticipantRole.UNKNOWN
    evidence: str | None = None


class ParticipantExtraction(BaseModel):
    participants: list[ProjectParticipant] = Field(default_factory=list)

In [64]:
# ============================================================
# 7. Activos energéticos
# ============================================================

class AssetRole(str, Enum):
    NEW_ASSET = "new_asset"
    EXISTING_ASSET = "existing_asset"
    MODIFIED_ASSET = "modified_asset"
    AFFECTED_ASSET = "affected_asset"
    ASSOCIATED_ASSET = "associated_asset"
    MAIN_ASSET = "main_asset"
    UNKNOWN = "unknown"


class AssetStatus(str, Enum):
    PLANNED = "planned"
    UNDER_PERMITTING = "under_permitting"
    AUTHORIZED = "authorized"
    UNDER_CONSTRUCTION = "under_construction"
    EXISTING = "existing"
    IN_OPERATION = "in_operation"
    DENIED = "denied"
    ARCHIVED = "archived"
    UNKNOWN = "unknown"


class EnergyAsset(BaseModel):
    local_asset_id: str
    name: str | None = None
    aliases: list[str] = Field(default_factory=list)

    role_in_event: AssetRole = AssetRole.UNKNOWN
    status_in_document: AssetStatus = AssetStatus.UNKNOWN

    technologies: list[Technology] = Field(default_factory=list)
    storage_systems: list[StorageSystem] = Field(default_factory=list)

    evidence: str | None = None


class AssetExtraction(BaseModel):
    assets: list[EnergyAsset] = Field(default_factory=list)

In [65]:
# ============================================================
# 8. Relaciones entre activos
# ============================================================

class AssetRelationType(str, Enum):
    HYBRIDIZES_WITH = "hybridizes_with"
    ADDS_TECHNOLOGY_TO = "adds_technology_to"
    MODIFIES = "modifies"
    EXPANDS = "expands"
    REPOWERS = "repowers"
    ADDS_STORAGE_TO = "adds_storage_to"
    SHARES_GRID_ACCESS_WITH = "shares_grid_access_with"
    SAME_PROJECT_GROUP_AS = "same_project_group_as"
    ASSOCIATED_WITH = "associated_with"
    UNKNOWN = "unknown"


class AssetRelation(BaseModel):
    source_asset_id: str
    target_asset_id: str
    relation_type: AssetRelationType
    evidence: str | None = None


class AssetRelationExtraction(BaseModel):
    asset_relations: list[AssetRelation] = Field(default_factory=list)

In [66]:
# ============================================================
# 9. Localizaciones extraídas
# ============================================================

class AssetLocationMention(BaseModel):
    local_asset_id: str | None = None
    raw_municipality: str
    raw_province: str | None = None
    raw_autonomous_community: str | None = None
    resolved_location: MunicipalityLocation | None = None
    evidence: str | None = None


class LocationExtraction(BaseModel):
    locations: list[AssetLocationMention] = Field(default_factory=list)

In [67]:
# ============================================================
# 10. Candidatos de agrupación
# ============================================================

class GroupingCandidateExtraction(BaseModel):
    grouping_candidates: list[str] = Field(default_factory=list)
    grouping_notes: str | None = None

In [68]:
# ============================================================
# 11. Documento BOE ensamblado
# ============================================================

class ProjectLifecycleEvent(BaseModel):
    event_type: LifecycleEventType = LifecycleEventType.UNKNOWN
    administrative_actions: list[AdministrativeAction] = Field(default_factory=list)

    assets: list[EnergyAsset] = Field(default_factory=list)
    asset_relations: list[AssetRelation] = Field(default_factory=list)

    event_summary: str | None = None
    evidence: str | None = None


class BOEProjectExtraction(BaseModel):
    identificador_boe: str
    fecha_publicacion: date | None = None

    relevancia_energetica: RelevanciaEnergetica
    es_relevante_para_proyecto: bool
    relevance_reason: str | None = None

    lifecycle_events: list[ProjectLifecycleEvent] = Field(default_factory=list)

    participants: list[ProjectParticipant] = Field(default_factory=list)
    locations: list[AssetLocationMention] = Field(default_factory=list)

    grouping_candidates: list[str] = Field(default_factory=list)
    grouping_notes: str | None = None

    extraction_notes: str | None = None

## 3. Agentes faseados

In [69]:
def build_agent(
    model,
    output_type: type[BaseModel],
    instructions: str,
    *,
    retries: int = 3,
) -> Agent:
    return Agent(
        model,
        output_type=output_type,
        instructions=instructions,
        retries=retries,
    )

In [70]:
# Para Ollama

from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider


def build_ollama_model(model_name: str) -> OllamaModel:
    return OllamaModel(
        model_name,
        provider=OllamaProvider(
            base_url="http://localhost:11434/v1",
        ),
    )

### Modelos disponibles

In [71]:
# MODEL_PROVIDER = "ollama"
MODEL_PROVIDER = "gemini"

In [72]:
if MODEL_PROVIDER == "gemini":
    AI_MODEL_NAME = "google:gemini-2.5-flash"
    AI_MODEL = AI_MODEL_NAME

elif MODEL_PROVIDER == "ollama":
    AI_MODEL_NAME = "qwen3:8b"
    AI_MODEL = build_ollama_model(AI_MODEL_NAME)

else:
    raise ValueError(
        f"Proveedor de modelo no soportado: {MODEL_PROVIDER}"
    )

### Agentes faseados

#### Classification

In [73]:
classification_agent = build_agent(
    model=AI_MODEL,
    output_type=DocumentClassification,
    instructions=CLASSIFICATION_INSTRUCTIONS,
)

#### Lifecycle

In [74]:
administrative_action_agent = build_agent(
    model=AI_MODEL,
    output_type=AdministrativeActionExtraction,
    instructions=ADMIN_EVENT_INSTRUCTIONS,
)

lifecycle_event_agent = build_agent(
    model=AI_MODEL,
    output_type=LifecycleEventExtraction,
    instructions=LIFECYCLE_EVENT_INSTRUCTIONS,
)

#### Assets

In [75]:
asset_agent = build_agent(
    model=AI_MODEL,
    output_type=AssetExtraction,
    instructions=ASSET_INSTRUCTIONS,
)

asset_relation_agent = build_agent(
    model=AI_MODEL,
    output_type=AssetRelationExtraction,
    instructions=ASSET_RELATION_INSTRUCTIONS,
)

#### Participants

In [76]:
participant_agent = build_agent(
    model=AI_MODEL,
    output_type=ParticipantExtraction,
    instructions=PARTICIPANT_INSTRUCTIONS,
)

#### Locations

In [77]:
location_agent = build_agent(
    model=AI_MODEL,
    output_type=LocationExtraction,
    instructions=LOCATION_INSTRUCTIONS,
)

##### Herramienta de location_agent

In [78]:
municipios_ine_df = pd.read_parquet(DIM_MUNICIPALITIES_PATH)

In [79]:
# Palabras con poco valor discriminante para identificar municipios.
STOP_TOKENS = {
    "a", "de", "del", "el", "en", "la", "las", "los", "y",
    "municipio", "municipal", "termino",
}


def text_tokens(text: str | None) -> set[str]:
    """
    Convierte un texto en un conjunto de tokens normalizados,
    eliminando palabras poco informativas.
    """
    return {
        token
        for token in normalize_text(text).split()
        if token not in STOP_TOKENS
    }


def token_overlap_score(query: str | None, candidate: str | None) -> float:
    """
    Calcula la proporción de tokens de la consulta presentes
    en el candidato.

    Valor entre 0 y 1.
    """
    query_tokens = text_tokens(query)
    candidate_tokens = text_tokens(candidate)

    if not query_tokens or not candidate_tokens:
        return 0.0

    return len(query_tokens & candidate_tokens) / len(query_tokens)


def municipality_token_matches(
    municipality_name: str,
    candidate_municipality: str,
    *,
    allow_single_token: bool,
) -> bool:
    """
    Determina si un municipio candidato es compatible con la consulta.

    Reglas:
    - Coincidencia total de tokens -> match.
    - Coincidencia >= 80 % para consultas con varios tokens -> match.
    - Consultas de un solo token solo se aceptan si existen hints
      adicionales (provincia o comunidad autónoma).
    """
    query_tokens = text_tokens(municipality_name)
    candidate_tokens = text_tokens(candidate_municipality)

    if not query_tokens or not candidate_tokens:
        return False

    if query_tokens.issubset(candidate_tokens):
        return True

    if len(query_tokens) >= 2:
        return token_overlap_score(municipality_name, candidate_municipality) >= 0.8

    return allow_single_token and bool(query_tokens & candidate_tokens)


def hint_token_matches(
    hint: str | None,
    candidate: str | None,
) -> bool:
    """
    Comprueba si un hint administrativo (provincia o comunidad autónoma)
    comparte al menos un token relevante con el candidato.
    """
    if hint is None:
        return True

    hint_tokens = text_tokens(hint)
    candidate_tokens = text_tokens(candidate)

    if not hint_tokens or not candidate_tokens:
        return False

    return bool(hint_tokens & candidate_tokens)


def _row_to_location(row: pd.Series) -> MunicipalityLocation:
    """
    Convierte una fila de la dimensión INE en un objeto tipado.
    """
    return MunicipalityLocation(
        municipality=row["municipio"],
        province=row["provincia"],
        autonomous_community=row["comunidad_autonoma"],
        ine_municipality_code=row["cpro"] + row["cmun"],
        ine_province_code=row["cpro"],
        ine_autonomous_community_code=row["cauto"],
    )


def _build_lookup_result(
    municipality_name: str,
    province_hint: str | None,
    autonomous_community_hint: str | None,
    matches: pd.DataFrame,
    matched_by: str,
    reason: str,
) -> MunicipalityLookupResult:
    """
    Construye la respuesta final a partir de las coincidencias obtenidas.

    - 1 coincidencia  -> RESOLVED
    - >1 coincidencia -> AMBIGUOUS
    - 0 coincidencias -> NOT_FOUND
    """
    matches = matches.drop_duplicates(
        subset=["cauto", "cpro", "cmun"]
    )

    if len(matches) == 1:
        return MunicipalityLookupResult(
            query=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.RESOLVED,
            resolved=_row_to_location(matches.iloc[0]),
            matched_by=matched_by,
            reason=reason,
        )

    if len(matches) > 1:
        return MunicipalityLookupResult(
            query=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.AMBIGUOUS,
            candidates=[_row_to_location(row) for _, row in matches.iterrows()],
            matched_by=matched_by,
            reason=(
                "Existen varias coincidencias compatibles con los criterios "
                "proporcionados."
            ),
        )

    return MunicipalityLookupResult(
        query=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        reason="No existe coincidencia en el catálogo INE.",
    )


def resolve_municipality_impl(
    municipality_name: str,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    # Normalizar consulta y hints para matching.
    municipality_name_norm = normalize_text(municipality_name)
    province_hint_norm = normalize_text(province_hint) if province_hint else None
    autonomous_community_hint_norm = (
        normalize_text(autonomous_community_hint)
        if autonomous_community_hint
        else None
    )

    # Fase 1: coincidencia exacta por nombre normalizado de municipio.
    matches = municipios_ine_df.loc[
        municipios_ine_df["municipio_norm"] == municipality_name_norm
    ]

    # Fase 2: desambiguar coincidencias exactas mediante provincia.
    if not matches.empty and province_hint_norm:
        province_matches = matches.loc[
            matches["provincia"].map(
                lambda value: hint_token_matches(province_hint, value)
            )
        ]

        if not province_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_matches,
                matched_by="municipality_exact_and_province_hint",
                reason=(
                    "Municipio resuelto por coincidencia exacta de municipio "
                    "y provincia compatible por tokens."
                ),
            )

    # Fase 3: desambiguar coincidencias exactas mediante comunidad autónoma.
    if not matches.empty and autonomous_community_hint_norm:
        ac_matches = matches.loc[
            matches["comunidad_autonoma"].map(
                lambda value: hint_token_matches(
                    autonomous_community_hint,
                    value,
                )
            )
        ]

        if not ac_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_matches,
                matched_by="municipality_exact_and_autonomous_community_hint",
                reason=(
                    "Municipio resuelto por coincidencia exacta de municipio "
                    "y comunidad autónoma compatible por tokens."
                ),
            )

    # Fase 4: si la coincidencia exacta ya es única, resolver.
    if not matches.empty:
        return _build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=matches,
            matched_by="municipality_exact",
            reason="Municipio resuelto por coincidencia exacta.",
        )

    # Permitir búsquedas de un solo token únicamente cuando existen hints.
    allow_single_token = (
        province_hint is not None
        or autonomous_community_hint is not None
    )

    # Fase 5: búsqueda flexible por tokens del municipio.
    partial_matches = municipios_ine_df.loc[
        municipios_ine_df["municipio"].map(
            lambda value: municipality_token_matches(
                municipality_name,
                value,
                allow_single_token=allow_single_token,
            )
        )
    ]

    # Fase 6: filtrar coincidencias parciales mediante provincia.
    if not partial_matches.empty and province_hint is not None:
        province_partial_matches = partial_matches.loc[
            partial_matches["provincia"].map(
                lambda value: hint_token_matches(province_hint, value)
            )
        ]

        if not province_partial_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_partial_matches,
                matched_by="municipality_token_and_province_hint",
                reason=(
                    "Municipio resuelto por coincidencia de tokens del municipio "
                    "y provincia compatible por tokens."
                ),
            )

    # Fase 7: filtrar coincidencias parciales mediante comunidad autónoma.
    if not partial_matches.empty and autonomous_community_hint is not None:
        ac_partial_matches = partial_matches.loc[
            partial_matches["comunidad_autonoma"].map(
                lambda value: hint_token_matches(
                    autonomous_community_hint,
                    value,
                )
            )
        ]

        if not ac_partial_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_partial_matches,
                matched_by="municipality_token_and_autonomous_community_hint",
                reason=(
                    "Municipio resuelto por coincidencia de tokens del municipio "
                    "y comunidad autónoma compatible por tokens."
                ),
            )

    # Fase 8: devolver coincidencias parciales restantes.
    if not partial_matches.empty:
        return _build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=partial_matches,
            matched_by="municipality_token",
            reason=(
                "Existen coincidencias por tokens del municipio, pero no hay "
                "hints suficientes para garantizar una resolución única."
            ),
        )

    # Fase 9: sin coincidencias exactas ni parciales.
    return MunicipalityLookupResult(
        query=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        reason="No existe coincidencia exacta ni por tokens en el catálogo INE.",
    )

In [80]:
@location_agent.tool_plain
def resolve_municipality(
    municipality_name: str,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    return resolve_municipality_impl(
        municipality_name=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
    )

# Tests opcionales
# resolve_municipality("Amurrio")
# resolve_municipality("Agurain/Salvatierra")
# resolve_municipality("Palmas")
# resolve_municipality("Palmas", province_hint="Las Palmas")
# resolve_municipality("Gran Canaria", province_hint="Las Palmas")
# resolve_municipality("Palmas", autonomous_community_hint="Canarias")

#### Grouping

In [81]:
grouping_agent = build_agent(
    model=AI_MODEL,
    output_type=GroupingCandidateExtraction,
    instructions=GROUPING_INSTRUCTIONS,
)

## 4. Pipeline de extracción faseada

In [82]:
# Construir prompt base

def build_boe_prompt(
    row: pd.Series,
    *,
    text_limit: int = 4000,  # 12000
) -> str:
    return f"""
Identificador BOE: {row["identificador"]}
Fecha publicación: {row["fecha_publicacion"]}
Título: {row["titulo"]}

Texto:
{row["texto_limpio"][:text_limit]}
"""

In [83]:
# Ejecutar una fase

async def run_extraction_phase(
    agent: Agent,
    prompt: str,
) -> BaseModel:
    result = await agent.run(prompt)
    return result.output

In [84]:
# Ensamblar la extracción final

def assemble_boe_project_extraction(
    *,
    classification: DocumentClassification,
    administrative_actions: AdministrativeActionExtraction,
    lifecycle_event: LifecycleEventExtraction,
    assets: AssetExtraction,
    asset_relations: AssetRelationExtraction,
    participants: ParticipantExtraction,
    locations: LocationExtraction,
    grouping: GroupingCandidateExtraction,
) -> BOEProjectExtraction:

    event = ProjectLifecycleEvent(
        event_type=lifecycle_event.event_type,
        administrative_actions=administrative_actions.administrative_actions,
        assets=assets.assets,
        asset_relations=asset_relations.asset_relations,
        event_summary=lifecycle_event.event_summary,
        evidence=lifecycle_event.evidence,
    )

    return BOEProjectExtraction(
        identificador_boe=classification.identificador_boe,
        fecha_publicacion=classification.fecha_publicacion,
        relevancia_energetica=classification.relevancia_energetica,
        es_relevante_para_proyecto=classification.es_relevante_para_proyecto,
        relevance_reason=classification.relevance_reason,
        lifecycle_events=[event],
        participants=participants.participants,
        locations=locations.locations,
        grouping_candidates=grouping.grouping_candidates,
        grouping_notes=grouping.grouping_notes,
        extraction_notes=None,
    )

In [85]:
# Extraer un documento

async def extract_single_boe_document(
    row: pd.Series,
    *,
    text_limit: int = 4000,  # 12000
) -> BOEProjectExtraction:
    prompt = build_boe_prompt(
        row,
        text_limit=text_limit,
    )

    classification = await run_extraction_phase(
        classification_agent,
        prompt,
    )

    if not classification.es_relevante_para_proyecto:
        return BOEProjectExtraction(
            identificador_boe=classification.identificador_boe,
            fecha_publicacion=classification.fecha_publicacion,
            relevancia_energetica=classification.relevancia_energetica,
            es_relevante_para_proyecto=False,
            relevance_reason=classification.relevance_reason,
            lifecycle_events=[],
            grouping_candidates=[],
            extraction_notes="Documento clasificado como no relevante para proyecto energético concreto.",
        )

    administrative_actions = await run_extraction_phase(
        administrative_action_agent,
        prompt,
    )

    lifecycle_event = await run_extraction_phase(
        lifecycle_event_agent,
        prompt,
    )

    assets = await run_extraction_phase(
        asset_agent,
        prompt,
    )

    asset_context = f"""
{prompt}

Activos identificados:
{assets.model_dump_json(indent=2)}
"""

    asset_relations = await run_extraction_phase(
        asset_relation_agent,
        asset_context,
    )

    participants = await run_extraction_phase(
        participant_agent,
        prompt,
    )

    locations = await run_extraction_phase(
        location_agent,
        prompt,
    )

    grouping_context = f"""
{prompt}

Clasificación:
{classification.model_dump_json(indent=2)}

Evento:
{lifecycle_event.model_dump_json(indent=2)}

Activos:
{assets.model_dump_json(indent=2)}

Participantes:
{participants.model_dump_json(indent=2)}

Localizaciones:
{locations.model_dump_json(indent=2)}
"""

    grouping = await run_extraction_phase(
        grouping_agent,
        grouping_context,
    )

    return assemble_boe_project_extraction(
        classification=classification,
        administrative_actions=administrative_actions,
        lifecycle_event=lifecycle_event,
        assets=assets,
        asset_relations=asset_relations,
        participants=participants,
        locations=locations,
        grouping=grouping,
    )

In [86]:
# Prueba

df = pd.read_parquet(BOE_CANDIDATES_DOCS_TEXT_PATH)


In [87]:
df_test = df.loc[df["xml_status"] == "ok"].copy()

df_test2 = df_test.loc[
    df_test["identificador"] == "BOE-A-2023-10306"
]

In [88]:
df_test2

,identificador,doc_file_stem,url_html,url_xml,fecha_publicacion,titulo,epigrafe_nombre,departamento_nombre,seccion_nombre,xml_path,texto_limpio,texto_len,xml_status,parse_error,parsed_at
87,BOE-A-2023-10306,20230428_BOE-A-2023-10306,https://www.boe.es/diario_boe/txt.php?id=BOE-A...,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...",Instalaciones eléctricas,MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL R...,III. Otras disposiciones,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,BOE-A-2023-10306 Estatal Ministerio para la Tr...,33226,ok,None,2026-06-15T10:38:10.041511+00:00


In [89]:
row = df_test2.iloc[0]

extraction = await extract_single_boe_document(row)

print(
    json.dumps(
        extraction.model_dump(),
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)

Traceback (most recent call last):
  File "/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-renewables-permitting-tracker/.venv/lib/python3.10/site-packages/pydantic_ai/models/google.py", line 821, in _generate_content
    return await func(model=self._model_name, contents=contents, config=config)  # type: ignore
  File "/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-renewables-permitting-tracker/.venv/lib/python3.10/site-packages/google/genai/models.py", line 8628, in generate_content
    return await self._generate_content(
  File "/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-renewables-permitting-tracker/.venv/lib/python3.10/site-packages/google/genai/models.py", line 7102, in _generate_content
    response = await self._api_client.async_request(
  File "/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-renewables-permitting-tracker/.venv/lib/python3.10/site-packages/google/genai/_api_client.py", line 1657, in async_request
    result = await self._async_request(
  File "/home/bgonzale/CiD